# Tree level 1 — first real tree layer on storage v2

Chunks -> clusters -> `nodes`/`edges` rows -> LLM node summaries. Everything reuses
frozen recipes:
- **Clustering**: the settled pipeline (α=0.5 combined content+path vectors, UMAP-25
  seed 42, HDBSCAN leaf mcs=5, noise chunks attached to nearest clustered chunk in the
  original combined space) — `build_clusters.cluster()` imported unchanged, fed with
  v2 chunk vectors (granite-97m, the experiment model).
- **Summarization**: the chosen two-layer gotcha-aware map-reduce from
  `summarization-experiments.md` — per-chunk Haiku condensation cached by content hash
  (same cache dir as the experiments — already-condensed content is free), then the
  `adaptive-gotchas` reduce (Haiku). Prompts + Claude Agent SDK backend imported from
  `summarize_experiments.py` unchanged.
- **References are mechanical**: edges rows, never LLM-written.

Scope: ONE layer, summaries for a 5-node probe (tightest / loosest / largest / median
coherence / seeded-random) — enough to eyeball quality before paying for all nodes.

In [1]:
import json
import sqlite3
import time
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

import build_clusters as bc

APP_DIR = Path.cwd()
DB_PATH = APP_DIR / ".hrono" / "hrono-v2.db"
GIT_CACHE = APP_DIR / ".hrono" / "cache" / "git"
MODEL_NAME = "ibm-granite/granite-embedding-97m-multilingual-r2"
ALPHA, BATCH = 0.5, 16

db = sqlite3.connect(DB_PATH)
model = SentenceTransformer(MODEL_NAME)

# chunk bodies from cache byte spans; embed whatever isn't in the embeddings table yet
chunk_rows = db.execute(
    "SELECT id, instance, path, byte_start, byte_end, hash, chunk_no, uri FROM chunks"
).fetchall()
body_by_hash = {}
for _, inst, path, s, e, h, _, _ in chunk_rows:
    if h not in body_by_hash:
        raw = (GIT_CACHE / inst / path).read_bytes()
        body_by_hash[h] = raw[s:e].decode("utf-8", "replace")

done = {r[0] for r in db.execute("SELECT hash FROM embeddings WHERE model = ?", (MODEL_NAME,))}
todo = [h for h in body_by_hash if h not in done]
print(f"{len(body_by_hash)} unique chunk bodies, {len(todo)} to embed")
for i in range(0, len(todo), BATCH):
    batch = todo[i:i + BATCH]
    vecs = model.encode([body_by_hash[h] for h in batch], batch_size=BATCH,
                        normalize_embeddings=True, show_progress_bar=False)
    with db:
        db.executemany("INSERT INTO embeddings (hash, model, dims, vector) VALUES (?,?,?,?)",
                       [(h, MODEL_NAME, vecs.shape[1], np.asarray(v, np.float32).tobytes())
                        for h, v in zip(batch, vecs)])

# one row per unique hash (dedup like the v1 pipeline); representative path per hash
rows = db.execute("""
    SELECT c.hash, c.path, e.vector FROM chunks c
    JOIN embeddings e ON e.hash = c.hash AND e.model = ?
    GROUP BY c.hash""", (MODEL_NAME,)).fetchall()
hashes = [r[0] for r in rows]
chunk_paths = [r[1] for r in rows]
content_vecs = np.frombuffer(b"".join(r[2] for r in rows), np.float32).reshape(len(rows), -1)

distinct_paths = sorted(set(chunk_paths))
pv = dict(zip(distinct_paths, model.encode(distinct_paths, batch_size=BATCH,
                                           normalize_embeddings=True, show_progress_bar=False)))
path_vecs = np.stack([pv[p] for p in chunk_paths])
combined = np.hstack([np.sqrt(1 - ALPHA) * content_vecs, np.sqrt(ALPHA) * path_vecs])
print(f"{len(hashes)} unique-hash chunks, combined dims {combined.shape[1]}")

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

501 unique chunk bodies, 0 to embed


501 unique-hash chunks, combined dims 768


## Cluster and materialize the layer: `nodes` + `edges`

Schema kept minimal for the first layer: a node is a cluster at a level; an edge binds a
node to its children (chunks at level 1; nodes at level 2+, hence child_kind). Summary
lands on the node row once generated — with the model and cost, because provenance of
generated text is data too.

In [2]:
labels, assigned, assignment_sim = bc.cluster(combined)

# summaries survive rebuilds: keyed by the node's member-set signature (sorted member
# hashes), the germ of the future diff layer — unchanged membership = reusable summary
try:
    saved = {r[0]: r[1:] for r in db.execute(
        "SELECT sig, summary, summary_model, summary_cost_usd, summary_prompt_id,"
        " summary_created_at FROM nodes WHERE summary IS NOT NULL")}
except sqlite3.OperationalError:
    try:  # pre-provenance schema: carry what exists
        saved = {r[0]: r[1:] + (None, None) for r in db.execute(
            "SELECT sig, summary, summary_model, summary_cost_usd FROM nodes"
            " WHERE summary IS NOT NULL")}
    except sqlite3.OperationalError:
        saved = {}
print(f"{len(saved)} existing summaries carried over by signature")

db.executescript("""
DROP TABLE IF EXISTS edges;
DROP TABLE IF EXISTS nodes;
CREATE TABLE nodes (
    id        INTEGER PRIMARY KEY,
    level     INTEGER NOT NULL,          -- 1 = clusters of chunks
    size      INTEGER NOT NULL,          -- member count
    coherence REAL    NOT NULL,          -- mean pairwise cosine of member content vectors
    sig       TEXT    NOT NULL,          -- sha256 of sorted member hashes: summary reuse key
    summary   TEXT,                      -- NULL until generated
    summary_model TEXT,
    summary_cost_usd REAL,
    summary_prompt_id INTEGER REFERENCES prompts(id),
    summary_created_at TEXT              -- ISO-8601 UTC
);
CREATE TABLE edges (
    parent_id  INTEGER NOT NULL REFERENCES nodes(id),
    child_kind TEXT    NOT NULL,         -- 'chunk' (level 1) | 'node' (level 2+)
    child_id   INTEGER NOT NULL,         -- chunks.id or nodes.id
    assigned   INTEGER NOT NULL,         -- 1 = noise chunk attached by proximity, not density
    UNIQUE (parent_id, child_kind, child_id)
);
""")

chunk_ids_by_hash = {}
for cid, inst, path, s, e, h, chunk_no, uri in chunk_rows:
    chunk_ids_by_hash.setdefault(h, []).append(cid)

with db:
    for label in sorted(set(labels)):
        idx = [i for i, l in enumerate(labels) if l == label]
        coherence = bc.mean_pairwise_cosine(content_vecs[idx])
        members = [(cid, bool(assigned[i])) for i in idx for cid in chunk_ids_by_hash[hashes[i]]]
        from hashlib import sha256 as _sha
        sig = _sha("".join(sorted(hashes[i] for i in idx)).encode()).hexdigest()
        summary, s_model, s_cost, s_prompt, s_created = saved.get(
            sig, (None, None, None, None, None))
        cur = db.execute(
            "INSERT INTO nodes (level, size, coherence, sig, summary, summary_model,"
            " summary_cost_usd, summary_prompt_id, summary_created_at)"
            " VALUES (1, ?, ?, ?, ?, ?, ?, ?, ?)",
            (len(members), coherence, sig, summary, s_model, s_cost, s_prompt, s_created))
        db.executemany("INSERT INTO edges (parent_id, child_kind, child_id, assigned)"
                       " VALUES (?, 'chunk', ?, ?)",
                       [(cur.lastrowid, cid, int(a)) for cid, a in members])

import pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 120)
nodes_df = pd.read_sql_query("""
    SELECT n.id, n.level, n.size, n.coherence,
           (SELECT GROUP_CONCAT(DISTINCT substr(c.path, 1, instr(c.path || '/', '/') +
                   instr(substr(c.path || '/', instr(c.path || '/', '/') + 1), '/') - 1))
            FROM edges e JOIN chunks c ON c.id = e.child_id
            WHERE e.parent_id = n.id LIMIT 3) AS repos
    FROM nodes n ORDER BY n.coherence DESC""", db)
print(f"{len(nodes_df)} level-1 nodes, sizes {sorted(nodes_df['size'].tolist(), reverse=True)}")
display(nodes_df)

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP: 768 -> 25 dims
HDBSCAN leaf: 41 clusters, 130 noise chunks to assign
4 existing summaries carried over by signature
41 level-1 nodes, sizes [35, 24, 22, 19, 19, 19, 18, 16, 16, 16, 15, 15, 15, 13, 12, 12, 12, 11, 10, 10, 10, 10, 10, 10, 9, 9, 9, 9, 9, 9, 9, 8, 8, 7, 7, 7, 7, 7, 6, 6, 6]


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


,id,level,size,coherence,repos
0,7,1,6,0.9398,branow/webphone
1,4,1,6,0.9366,branow/webphone
2,3,1,7,0.9301,branow/football-match-mvc
3,23,1,7,0.9247,branow/webphone
4,32,1,12,0.9242,branow/webphone
5,33,1,9,0.9226,branow/webphone
6,2,1,8,0.9191,branow/webphone
7,1,1,16,0.9158,branow/webphone
8,31,1,7,0.9150,branow/webphone
9,13,1,12,0.9135,branow/football-match-mvc


## Summarize the 5-node probe

Recipe verbatim from the experiments: map = `condense_chunk` (Haiku, GOTCHA lines,
cached by content hash in the shared map-cache), reduce = `adaptive-gotchas` prompt
(Haiku) over the condensations. Chunk labels are real paths so the summary can name
files. Probe: tightest, loosest, largest, median-coherence, seeded-random.

In [3]:
import random
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

import summarize_experiments as se

from datetime import datetime, timezone

# prompts are data: every generated text references the exact prompt that produced it.
# (name, text) unique — editing a prompt creates a NEW row; history stays truthful.
db.executescript("""
CREATE TABLE IF NOT EXISTS prompts (
    id   INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    text TEXT NOT NULL,
    UNIQUE (name, text)
);
CREATE TABLE IF NOT EXISTS condensations (
    hash       TEXT PRIMARY KEY,   -- chunk content hash
    model      TEXT NOT NULL,
    text       TEXT NOT NULL,
    prompt_id  INTEGER REFERENCES prompts(id),
    created_at TEXT                -- ISO-8601 UTC
);
""")
for col, ddl in (("prompt_id", "INTEGER REFERENCES prompts(id)"), ("created_at", "TEXT")):
    if col not in {r[1] for r in db.execute("PRAGMA table_info(condensations)")}:
        db.execute(f"ALTER TABLE condensations ADD COLUMN {col} {ddl}")


def prompt_id(name, text):
    with db:
        db.execute("INSERT OR IGNORE INTO prompts (name, text) VALUES (?,?)", (name, text))
    return db.execute("SELECT id FROM prompts WHERE name = ? AND text = ?",
                      (name, text)).fetchone()[0]


def utcnow():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


# verbatim copy of se.condense_chunk's inline template (that file is experiment
# history — not edited; keep this in sync manually if the recipe ever changes)
MAP_PROMPT = (
    "Condense this content chunk to its essential information. Let the length "
    "follow the density of the content: boilerplate, generated, or repetitive "
    "content collapses to one short line saying so; dense logic keeps whatever "
    "it needs to stay faithful. Copy concrete identifiers, names, values, and "
    "constraints VERBATIM from the content - never infer or add ones that are "
    "not literally there. If the content holds anything non-obvious a newcomer "
    "could be hurt by (behavior differing from what names suggest, silent "
    "no-ops, partial or non-transactional operations, asymmetries, workarounds, "
    "fragile special cases), add a line starting with 'GOTCHA:' for each - only "
    "if literally present. Output only the condensed content."
)
MAP_PROMPT_ID = prompt_id("map-condense", MAP_PROMPT)
REDUCE_PROMPT_ID = prompt_id("adaptive-gotchas", se.PROMPTS["adaptive-gotchas"])

# backfill provenance on rows created before these columns existed
with db:
    db.execute("UPDATE condensations SET prompt_id = ?, created_at = COALESCE(created_at, ?)"
               " WHERE prompt_id IS NULL", (MAP_PROMPT_ID, utcnow()))
    db.execute("UPDATE nodes SET summary_prompt_id = ?,"
               " summary_created_at = COALESCE(summary_created_at, ?)"
               " WHERE summary IS NOT NULL AND summary_prompt_id IS NULL",
               (REDUCE_PROMPT_ID, utcnow()))


def condense_all(chunks):
    """DB-first map stage. SQLite is main-thread-only here: lookups and inserts stay on
    this thread; only the Claude calls (misses) go to the pool."""
    out, missing = {}, []
    for c in chunks:
        row = db.execute("SELECT text FROM condensations WHERE hash = ?", (c.hash,)).fetchone()
        if row:
            out[c.label] = row[0]
        else:
            missing.append(c)
    if missing:
        with ThreadPoolExecutor(max_workers=se.PARALLEL_CALLS) as pool:
            results = list(pool.map(se.condense_chunk, missing))  # claude or se's file cache
        out.update({c.label: t for c, t in zip(missing, results)})
        with db:
            db.executemany(
                "INSERT OR IGNORE INTO condensations (hash, model, text, prompt_id, created_at)"
                " VALUES (?,?,?,?,?)",
                [(c.hash, se.MAP_MODEL, t, MAP_PROMPT_ID, utcnow())
                 for c, t in zip(missing, results)])
    print(f"  map: {len(chunks) - len(missing)} from DB, {len(missing)} new calls")
    return out


def node_chunks(node_id):
    return db.execute("""
        SELECT c.id, c.path, c.chunk_no, c.hash, c.instance, c.byte_start, c.byte_end
        FROM edges e JOIN chunks c ON c.id = e.child_id
        WHERE e.parent_id = ? ORDER BY c.path, c.chunk_no""", (node_id,)).fetchall()


@dataclass
class C:  # duck-typed stand-in for summarize_experiments.Chunk
    label: str
    hash: str
    text: str


def chunks_for_summary(node_id):
    out = []
    for cid, path, chunk_no, h, inst, s, e in node_chunks(node_id):
        raw = (GIT_CACHE / inst / path).read_bytes()
        out.append(C(label=f"{path}#chunk-{chunk_no}", hash=h,
                     text=raw[s:e].decode("utf-8", "replace")))
    return out


ordered = nodes_df.sort_values("coherence")
rng = random.Random(42)
probe = sorted({int(ordered.iloc[-1]["id"]),                       # tightest
                int(ordered.iloc[0]["id"]),                        # loosest
                int(nodes_df.sort_values("size").iloc[-1]["id"]),  # largest
                int(ordered.iloc[len(ordered) // 2]["id"]),        # median coherence
                int(rng.choice(nodes_df["id"].tolist()))})         # seeded random
print(f"probe nodes: {probe}")

for node_id in probe:
    if db.execute("SELECT summary FROM nodes WHERE id = ?", (node_id,)).fetchone()[0]:
        print(f"node {node_id}: summary exists, skipping")
        continue
    chunks = chunks_for_summary(node_id)
    print(f"node {node_id}: {len(chunks)} chunks — map stage...")
    condensed = condense_all(chunks)
    content = se.render_condensed(chunks, condensed)
    t0 = time.monotonic()
    # call_claude uses asyncio.run, illegal on Jupyter's event-loop thread -> worker thread
    with ThreadPoolExecutor(max_workers=1) as ex:
        result = ex.submit(se.call_claude,
                           se.instructions("adaptive-gotchas") + "\n\n" + content,
                           "haiku").result()
    with db:
        db.execute("UPDATE nodes SET summary = ?, summary_model = ?, summary_cost_usd = ?,"
                   " summary_prompt_id = ?, summary_created_at = ? WHERE id = ?",
                   (result["text"], "haiku:adaptive-gotchas-condensed", result["cost_usd"],
                    REDUCE_PROMPT_ID, utcnow(), node_id))
    print(f"  reduce done in {time.monotonic() - t0:.0f}s, ${result['cost_usd']:.3f}")

probe nodes: [6, 7, 22, 36]
node 6: summary exists, skipping
node 7: summary exists, skipping
node 22: summary exists, skipping
node 36: summary exists, skipping


## The summaries — eyeball review

Each probe node: its stats, member listing (mechanical, from edges), then the generated
summary. Judge: does it state what the group IS, use real names, compress repetition,
and surface believable gotchas?

In [4]:
for node_id in probe:
    size, coherence, summary, cost = db.execute(
        "SELECT size, coherence, summary, summary_cost_usd FROM nodes WHERE id = ?",
        (node_id,)).fetchone()
    members = node_chunks(node_id)
    paths = sorted({m[1] for m in members})
    listing = "\n".join(f"- `{p}`" for p in paths[:15])
    if len(paths) > 15:
        listing += f"\n- … {len(paths) - 15} more files"
    display(Markdown(
        f"---\n## node {node_id} — {size} chunks, coherence {coherence:.3f}, "
        f"cost ${cost or 0:.3f}\n**Members ({len(paths)} files):**\n{listing}\n\n"
        f"**Summary:**\n\n{summary or '(none)'}"))

total = db.execute("SELECT COUNT(*), SUM(summary_cost_usd) FROM nodes WHERE summary IS NOT NULL").fetchone()
print(f"\nsummarized {total[0]} nodes, total ${total[1] or 0:.3f}")
db.close()

---
## node 6 — 9 chunks, coherence 0.874, cost $0.020
**Members (9 files):**
- `branow/webphone/client/src/util/log.ts`
- `branow/webphone/server/src/main/java/com/scisbo/webphone/config/LoggingConfig.java`
- `branow/webphone/server/src/main/java/com/scisbo/webphone/log/config/DefaultLoggingAutoConfiguration.java`
- `branow/webphone/server/src/main/java/com/scisbo/webphone/log/config/LoggingConfiguration.java`
- `branow/webphone/server/src/main/java/com/scisbo/webphone/log/web/LoggingFilter.java`
- `branow/webphone/server/src/main/resources/logback.xml`
- `branow/webphone/server/src/test/java/com/scisbo/webphone/TestLoggingConfig.java`
- `branow/webphone/server/src/test/java/com/scisbo/webphone/log/id/RequestLogIdProviderTest.java`
- `branow/webphone/server/src/test/java/com/scisbo/webphone/log/web/LoggingFilterTest.java`

**Summary:**

# Logging System (Client & Server)

A comprehensive logging setup for the webphone application, spanning TypeScript client code and Java/Spring Boot backend, with request/response tracing and file rotation.

## Client-Side Logging

Simple conditional logging utility in `log.ts` that exports `warn()` and `log()` functions. Both functions only execute their console output when the environment variable `WEBPHONE_PROFILE === "dev"`; otherwise they are no-ops. Used for development debugging without affecting production builds.

## Server-Side Spring Configuration

Three interdependent configuration classes provide the logging infrastructure:

- **LoggingConfig**: Main @Configuration bean that defines `LoggingConfiguration` (with `RequestLogIdProvider` dependency) and `ConversionService` (with Page/Pageable converters)
- **DefaultLoggingAutoConfiguration**: Boot auto-config providing fallback beans when not explicitly defined—includes `NoneLogIdProvider`, `SpelExpressionParser`, `DefaultConversionService`, and conditional `LoggingFilter` registration
- **LoggingConfiguration**: Simple data class holding a `LogIdProvider` to generate unique IDs per log record

## Request/Response Logging

**LoggingFilter** (Jakarta servlet Filter) wraps HTTP interactions:
- Logs all requests at INFO level with format: `"Request: {METHOD} {URI}?{query}"`
- Logs responses with status-code-dependent levels: INFO for <400, WARN for 4xx, ERROR for 5xx
- Includes response body in error/warning logs (4xx+) only; successful responses log status code alone
- Uses `ContentCachingResponseWrapper` to capture response body without consuming the stream
- Initializes `SpelLogger` lazily on first request

## File and Console Output

Logback configuration (`logback.xml`) defines two appenders:
- **STDOUT**: ConsoleAppender for immediate console output
- **ROLLING_FILE**: SizeAndTimeBasedRollingPolicy writing to `logs/webphone.log`, rolling daily and by size (10MB per file), keeping 30 rollover files
- Both use pattern: `%d{HH:mm:ss.SSS} %-5level %logger{36} - %msg%n`
- Root logger level set to INFO

## Tests

Two test suites verify server-side logging:
- **RequestLogIdProviderTest**: Confirms ID consistency across a single request lifecycle (before filter, during handler, after filter)
- **LoggingFilterTest**: Verifies request/response logging behavior with mock `SpelLoggerFactory`, testing status codes 200/400/404/500 and confirming appropriate log levels and body inclusion

## Caution

- **LoggingFilter enabled by default**: `@ConditionalOnProperty(name = "log.filter.enabled", matchIfMissing=true)` means it runs unless explicitly set to `false` in properties
- **Response bodies selectively logged**: Only included in 4xx+ responses; 2xx responses log status code only
- **Missing endpoints show `[empty]`**: 404 responses display this placeholder instead of an actual body
- **File retention counts files, not days**: `maxHistory=30` retains 30 rollover files regardless of span; actual retention depends on files-per-day (high-volume days may roll multiple files, compressing the time window)
- **Logging level non-configurable per response**: Status-code-based levels are hardcoded in the filter; caller has no say in whether an endpoint's response logs at INFO vs. WARN vs. ERROR

---
## node 7 — 6 chunks, coherence 0.940, cost $0.017
**Members (6 files):**
- `branow/webphone/server/src/main/java/com/scisbo/webphone/log/core/SimpleSpelLoggerContext.java`
- `branow/webphone/server/src/main/java/com/scisbo/webphone/log/core/SpelLogger.java`
- `branow/webphone/server/src/main/java/com/scisbo/webphone/log/core/SpelLoggerContext.java`
- `branow/webphone/server/src/main/java/com/scisbo/webphone/log/core/SpelLoggerFactory.java`
- `branow/webphone/server/src/test/java/com/scisbo/webphone/log/config/DefaultLoggingAutoConfigurationTest.java`
- `branow/webphone/server/src/test/java/com/scisbo/webphone/log/core/SpelLoggerTest.java`

**Summary:**

# SpEL-Based Dynamic Logging Framework

A Spring-integrated logging system that evaluates Spring Expression Language (SpEL) templates to generate dynamic log messages with contextual variables (arguments, results, errors) and optional log ID prefixes.

## Core Components

**SpelLoggerContext** (interface) and **SimpleSpelLoggerContext** (implementation) manage the evaluation context for SpEL:
- Expose a wrapped `StandardEvaluationContext` for SpEL expression evaluation
- Provide setters to bind variables into the context: `setArguments(Map)`, `setResult()`, `setError()`

**SpelLogger** is the main logger class that:
- Takes a SpEL template string and log level as input to the `log()` method
- Parses the template using Spring's TemplateParserContext to embed expressions
- Evaluates the parsed expression against the EvaluationContext
- Logs the resulting message, optionally prepending `[logId]` prefix if a LogIdProvider returns a non-empty ID

## Factory & Configuration

**SpelLoggerFactory** is a Spring `@Component` that creates SpelLogger instances via three overloaded `getLogger()` methods:
- `getLogger(Class)` — uses SimpleSpelLoggerContext and DefaultLogIdProvider
- `getLogger(Class, LogIdProvider)` — accepts a custom ID provider
- `getLogger(Class, SpelLoggerContext, LogIdProvider)` — full control over context and provider

All variants also wrap the evaluation context's type converter with Spring's ConversionService.

Auto-configuration tests verify that the framework respects both Spring defaults (NoneLogIdProvider, default SpelExpressionParser) and custom bean overrides.

## Test Coverage

Unit tests cover:
- SpelLogger's expression parsing and log formatting, with both non-empty and empty log ID scenarios
- Spring auto-configuration's ability to provide defaults and honor custom bean overrides via @TestConfiguration

## Caution

- **Silent LogIdProvider replacement:** SpelLoggerFactory silently replaces any `DefaultLogIdProvider` parameter with the one from LoggingConfiguration, discarding the passed value. Only non-default providers are honored.
- **ID prefix asymmetry:** An empty string log ID suppresses the `[ID] ` prefix entirely; a non-empty ID always prepends `[ID] ` with a space, changing the message format.

---
## node 22 — 35 chunks, coherence 0.843, cost $0.038
**Members (35 files):**
- `branow/webphone/client/src/components/common/Hover.tsx`
- `branow/webphone/client/src/components/common/input/CheckBox.tsx`
- `branow/webphone/client/src/components/common/input/FileInput.tsx`
- `branow/webphone/client/src/components/common/input/OptionSet.tsx`
- `branow/webphone/client/src/components/common/input/SearchBar.tsx`
- `branow/webphone/client/src/components/common/input/Select.tsx`
- `branow/webphone/client/src/components/common/input/TextArea.tsx`
- `branow/webphone/client/src/components/common/input/TextInput.tsx`
- `branow/webphone/client/src/components/common/input/Toggler.tsx`
- `branow/webphone/client/src/components/common/messages/BackgroundMessage.tsx`
- `branow/webphone/client/src/components/common/messages/ErrorBanner.tsx`
- `branow/webphone/client/src/components/common/messages/ErrorMessage.tsx`
- `branow/webphone/client/src/components/common/pane/Chapter.tsx`
- `branow/webphone/client/src/components/common/pane/ChapterBar.tsx`
- `branow/webphone/client/src/components/common/pane/ControlPane.tsx`
- … 20 more files

**Summary:**

# Webphone Client UI Component Library

This is a React-based component library for a webphone application, comprising styled UI components, theme management, input controls, layout containers, navigation elements, and settings pages. The codebase uses Linaria for CSS-in-JS styling, React Context for theming, and i18n for internationalization.

## Theming & Styling Foundation

**Theme System**: `ThemeContext` provides `theme` and `setTheme`, with a `useTheme()` hook for component access. `ThemeProvider` (in `providers/`) persists theme choice to localStorage and respects system dark-mode preference. Two themes defined in `styles.ts`: `LightTheme` (light background, purple/orange navbar) and `DarkTheme` (dark background, burgundy/rust navbar).

**Global Styles**: `styles.ts` defines font sizes (s through x5l, 12–40px), layout dimensions (phone 300×580px, navbar/tabs/headers 50px height), color palettes, and animations (e.g., `pulsatingShaking` rotation/scale keyframes).

## Layout & Container Components

**Phone Mockup**: `PhoneContainer` wraps content in nested divs simulating a phone viewport (full 100vw/100vh outer, phone-sized inner with rounded borders and shadow). `PhoneLayout` orchestrates three zones: fixed `NavBar` header, `CallBar` controls, and `MainContainer` for main content (height calculated by subtracting navbar from phone height). Both delegate styling to Linaria.

**Modal Windows**: `Window` (full-screen centered overlay with background dismiss) and `RemoveWindow` (confirmation dialog) wrap content with styled containers, optional close buttons, and `ControlPane` for button controls.

**Section Containers**: `Chapter` (section with optional title), `ChapterBar` (horizontal divider), `Paragraph` (titled text block), `ControlPane` (flex row of uppercase bold buttons).

## Navigation & Header Components

**Navigation Bar**: `NavBar` centers three children—`ErrorWarning`, `NavBarTitle`, and `SettingButton`—in a surface-colored container. `NavBarTitle` renders "Webphone" text as a gradient-colored link to "/dialpad" with drop-shadow on hover.

**Navigation Tabs**: `NavTabs` renders clickable controls mapping to routes, positioned absolutely at the bottom of the phone (requires positioned parent). File named "NabTabs.tsx" (typo).

**Error Warning**: `ErrorWarning` displays a pulsing icon in the navbar that changes color by priority (red ≥20, yellow 10–19, gray <10). On hover, shows error messages sorted by priority; dismissible by click.

## Input & Form Components

**Text Inputs**: `TextInput` and `TextArea` are controlled components with optional labels, error messages (fade-animated), icons, and theme-aware styling (transparent background, bottom border only). Both fire dual callbacks: standard `onChange` (if provided) and a required `onValueChange`.

**Selection Controls**: 
- `CheckBox`: circular checkbox with yellow/gray styling based on checked state
- `Toggler`: switch component with sliding knob, customizable size and color
- `Select`: generic dropdown with selected option highlighted, filtered unselected list below
- `OptionSet`: horizontal flex row of options with bottom border on selected item
- `SearchBar`: text input with optional 300ms debounced callback

**File & Utility**: `FileInput` (hidden input triggered via ref, converts FileList to File[] array), `Hover` (render-props wrapper tracking mouse enter/leave state).

## Message & Feedback Components

**Error Display**: `ErrorMessage` and `ErrorBanner` extract error text (from Error.message or string) and render in red-hover color. `ErrorBanner` adds fade animation. Both return empty fragments when no error provided.

**General Messages**: `BackgroundMessage` renders centered bold text in disabled color.

**Error Pages**: `ErrorPage` (generic error display component despite being named `AccountNotFoundPage`) renders centered header, message, and action button.

## Settings Pages

**Theme Configuration**: `ThemeSetting` renders a label and `ThemeToggler` (animated toggle between light/dark themes). `ThemeToggler` displays LightTheme/DarkTheme icons, with icon always colored to theme.colors.yellow.

**Language Configuration**: `LanguageSetting` renders a dropdown selector for three languages ("en", "es", "uk") with country flag icons mapped to language codes (US, Spain, Ukraine).

**General Settings**: `GeneralSetting` wraps `ThemeSetting` and `LanguageSetting` in a flex column.

## Miscellaneous Components

**Keypad**: Phone keypad grid (4×3) with digits 1–9 plus phone letter labels (ABC, DEF, etc.), *, 0, #. Fires `onPressKey` callback with key objects.

---

## Caution

- **Dual callback pattern**: `TextInput`, `TextArea` fire BOTH `onChange` and the required `onValueChange`—components depend on explicit `onValueChange` prop, not standard React convention.
- **Component naming mismatches**: `ErrorPage` is generically named but exported from a file suggesting specific use; `RemoveWindow.tsx` exports a component called `DeleteWindow`; `NabTabs.tsx` has a filename typo.
- **Render-props gotchas**: `Hover` component overwrites `onMouseEnter`/`onMouseLeave` on children and assumes children's returned element has a `.type` property (fails on Fragments).
- **ThemeProvider bugs**: Line 18 sets `LightTheme` when system dark preference is detected (likely should be `DarkTheme`); event listener cleanup on line 26 doesn't actually unregister.
- **Layout fragility**: `PhoneLayout` height calculation (`size.phone.h - size.navbar.h`) breaks if size values desynchronize. Both `<Outlet />` and `children` render if both provided (not exclusive).
- **Select/OptionSet edge cases**: Both throw if options array is empty. `Select.handleSelect` toggles menu closed after every selection.
- **Theme color duplication**: `styles.ts` defines yellow, blue, red colors identically in light and dark themes—likely unintentional.
- **Error visibility**: `ErrorWarning` messages only appear on hover despite pulsing icon advertising their presence.
- **Priority inversion**: `ErrorWarning` colors decrease in urgency as numeric priority decreases (lower number = less alarming color).

---
## node 36 — 19 chunks, coherence 0.783, cost $0.026
**Members (19 files):**
- `branow/webphone/README.md`
- `branow/webphone/client/.gitignore`
- `branow/webphone/client/.onedev-buildspec.yml`
- `branow/webphone/client/README.md`
- `branow/webphone/client/eslint.config.js`
- `branow/webphone/client/index.html`
- `branow/webphone/client/package.json`
- `branow/webphone/client/public/phone.svg`
- `branow/webphone/client/public/silent-check-sso.html`
- `branow/webphone/client/src/declarations.d.ts`
- `branow/webphone/client/src/index.css`
- `branow/webphone/client/src/lib/storage.ts`
- `branow/webphone/client/src/vite-env.d.ts`
- `branow/webphone/client/tsconfig.app.json`
- `branow/webphone/client/tsconfig.json`
- … 4 more files

**Summary:**

# Webphone: React + Spring Boot Softphone System

This group documents a web-based softphone application built with React (frontend), Spring Boot (backend), Keycloak (auth), Asterisk PBX (VoIP), and MongoDB (data store). The frontend communicates directly with Asterisk via WebRTC-over-WSS for SIP signaling; the backend handles user authentication, validation, and data persistence only.

## Architecture & Core Documentation

The system is service-oriented: React frontend makes OAuth2 calls to Keycloak and REST calls to the backend (over HTTPS + JWT), while simultaneously establishing WebRTC-WSS connections directly to Asterisk. The backend never participates in SIP signaling and accesses MongoDB exclusively with per-user data isolation enforced via JWT context. Core features include user auth, SIP calling (make/receive/mute/hold/dial), address book with photos, call history, theme/language settings, and SIP credential management.

## Frontend Build & Configuration

The client is a TypeScript/React app using Vite (build), ESLint (linting), and strict TypeScript settings. Configuration includes:
- **package.json:** React 18, TanStack Query, Keycloak-JS, JsSIP, Framer Motion, i18next, Linaria CSS-in-JS, React Router, React Icons
- **tsconfig files:** ES2020/ESNext target with strict mode, no unused locals/parameters, react-jsx transformation
- **vite.config.ts:** Exposes 7 environment variables (WEBPHONE_PROFILE, WEBPHONE_CONTEXT_PATH, WEBPHONE_BACKEND_ORIGIN, WEBPHONE_AUTH_HEADERS_URL, WEBPHONE_KEYCLOAK_*), configures Babel presets for TypeScript/React, and path aliases for common directories (components, hooks, lib, pages, etc.)
- **ESLint config:** Extends TypeScript/React best practices with rules for React hooks and refresh-only exports

## Frontend Source & Utilities

- **index.html:** Standard entry point; mounts React app at `#root`, references `/phone.svg` icon
- **index.css:** Body reset with "Source Sans Pro" font
- **storage.ts:** Generic localStorage wrapper class with `set()`, `get()`, and `remove()` methods
- **declarations.d.ts, vite-env.d.ts:** TypeScript module declarations for CSS/JS imports and Vite client types
- **.gitignore:** Standard Node.js exclusions (node_modules, dist, logs, editor files)

## Deployment & CI/CD

- **.onedev-buildspec.yml:** OneDev CI pipeline (v38) triggered on master branch: sets legacy-peer-deps, runs `npm install` and `npm run build`, then copies dist files to `/home/onedev/bodnar/bodnar.scisbo.com/`. Retries up to 3 times with 30s delay; 14400s timeout.

## Authentication Helpers & Assets

- **silent-check-sso.html:** Iframe target for Keycloak silent SSO checks; posts current location to parent window
- **phone.svg:** 800×800px SVG phone icon (green/teal/beige gradient) with signal ripples

## Caution

**Frontend SIP signaling is direct:** The frontend connects directly to Asterisk over WebRTC-WSS for all SIP operations. The backend never sees SIP traffic—it is not a proxy or intermediary. This differs from traditional monolithic softphone architectures where the backend would mediate all calls.

**MongoDB access is backend-only:** The frontend never queries MongoDB directly. All data (contacts, call history, photos) flows through the REST API, which enforces per-user isolation via JWT context. Direct frontend-to-database access is not supported by design.


summarized 4 nodes, total $0.100


## Observations

(To fill after eyeball review. Open items carried in: Sonnet-for-messy-clusters heuristic
untested; level-2 composition next; the diff layer — only changed-membership nodes
re-summarize — is the update mechanism's job once the layer stabilizes.)